# MLExtend asso rules mining

In [18]:
import pandas as pd
import numpy as np
import re
from typing import NamedTuple
import ast

## Data

In [19]:
data_path = "data"

In [20]:
df_wine = pd.read_csv(f'{data_path}/WineDataset.csv')
df_X = pd.read_csv(f'{data_path}/XWines_Slim_1K_wines.csv')

In [21]:
df_X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   WineID      1007 non-null   int64  
 1   WineName    1007 non-null   object 
 2   Type        1007 non-null   object 
 3   Elaborate   1007 non-null   object 
 4   Grapes      1007 non-null   object 
 5   Harmonize   1007 non-null   object 
 6   ABV         1007 non-null   float64
 7   Body        1007 non-null   object 
 8   Acidity     1007 non-null   object 
 9   Code        1007 non-null   object 
 10  Country     1007 non-null   object 
 11  RegionID    1007 non-null   int64  
 12  RegionName  1007 non-null   object 
 13  WineryID    1007 non-null   int64  
 14  WineryName  1007 non-null   object 
 15  Website     900 non-null    object 
 16  Vintages    1007 non-null   object 
dtypes: float64(1), int64(3), object(13)
memory usage: 133.9+ KB


In [22]:
df_X.head()

,WineID,WineName,Type,Elaborate,Grapes,Harmonize,ABV,Body,Acidity,Code,Country,RegionID,RegionName,WineryID,WineryName,Website,Vintages
0,100001,Espumante Moscatel,Sparkling,Varietal/100%,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",7.5,Medium-bodied,High,BR,Brazil,1001,Serra Gaúcha,10001,Casa Perini,http://www.vinicolaperini.com.br,"[2020, 2019, 2018, 2017, 2016, 2015, 2014, 201..."
1,100002,Ancellotta,Red,Varietal/100%,['Ancellotta'],"['Beef', 'Barbecue', 'Codfish', 'Pasta', 'Pizz...",12.0,Medium-bodied,Medium,BR,Brazil,1001,Serra Gaúcha,10001,Casa Perini,http://www.vinicolaperini.com.br,"[2016, 2015, 2014, 2013, 2012, 2011, 2010, 200..."
2,100003,Cabernet Sauvignon,Red,Varietal/100%,['Cabernet Sauvignon'],"['Beef', 'Lamb', 'Poultry']",12.0,Full-bodied,High,BR,Brazil,1001,Serra Gaúcha,10002,Castellamare,https://www.emporiocastellamare.com.br,"[2021, 2020, 2019, 2018, 2017, 2016, 2015, 201..."
3,100005,Maison de Ville Cabernet-Merlot,Red,Assemblage/Bordeaux Red Blend,"['Cabernet Sauvignon', 'Merlot']","['Beef', 'Lamb', 'Game Meat', 'Poultry']",11.0,Full-bodied,Medium,BR,Brazil,1001,Serra Gaúcha,10000,Aurora,http://www.vinicolaaurora.com.br,"[2021, 2020, 2019, 2018, 2017, 2016, 2015, 201..."
4,100007,Do Lugar Moscatel Espumantes,Sparkling,Varietal/100%,['Muscat/Moscato Bianco'],"['Pork', 'Rich Fish', 'Shellfish']",7.5,Medium-bodied,High,BR,Brazil,1001,Serra Gaúcha,10012,Dal Pizzol,http://www.dalpizzol.com.br,"[2018, 2017, 2016, 2015, 2014, 2013, 2012, 201..."


In [23]:
df_X.Harmonize[0]

"['Pork', 'Rich Fish', 'Shellfish']"

In [24]:
df_X["Harmonize"] = df_X["Harmonize"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [25]:
df_X["Grapes"] = df_X["Grapes"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [26]:
df_X["Harmonize"][0]

['Pork', 'Rich Fish', 'Shellfish']

In [27]:
acid_dummies = pd.get_dummies(df_X["Acidity"], prefix="Acid")
body_dummies = pd.get_dummies(df_X["Body"], prefix="Body")

In [28]:
type_dummies = pd.get_dummies(df_X["Type"], prefix="Type")
harm_df = df_X[["WineID", "Harmonize"]].explode("Harmonize")
harm_dummies = pd.get_dummies(harm_df["Harmonize"], prefix="Pair")

harm_matrix = pd.concat([harm_df["WineID"], harm_dummies], axis=1)
harm_matrix = harm_matrix.groupby("WineID").max().reset_index()

In [29]:
harm_matrix["Pair_Aperitif"] = harm_matrix["Pair_Appetizer"] + harm_matrix["Pair_Aperitif"]
harm_matrix = harm_matrix.drop(columns=["Pair_Appetizer"])

In [30]:
harm_matrix.head()

,WineID,Pair_Aperitif,Pair_Barbecue,Pair_Beef,Pair_Blue Cheese,Pair_Cake,Pair_Cheese,Pair_Chicken,Pair_Chocolate,Pair_Codfish,...,Pair_Seafood,Pair_Shellfish,Pair_Snack,Pair_Soft Cheese,Pair_Soufflé,Pair_Spicy Food,Pair_Sweet Dessert,Pair_Tomato Dishes,Pair_Veal,Pair_Vegetarian
0,100001,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
1,100002,False,True,True,False,False,True,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,100003,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,100005,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,100007,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False


In [31]:
dataset_type_harm = (
    df_X[["WineID"]]
    .merge(type_dummies, left_index=True, right_index=True)
    .merge(harm_matrix, on="WineID")
)

dataset_type_harm = dataset_type_harm.set_index("WineID")

In [33]:
type_dummies.head()

,Type_Dessert,Type_Dessert/Port,Type_Red,Type_Rosé,Type_Sparkling,Type_White
0,False,False,False,False,True,False
1,False,False,True,False,False,False
2,False,False,True,False,False,False
3,False,False,True,False,False,False
4,False,False,False,False,True,False


In [34]:
print(df_X.columns.tolist())

['WineID', 'WineName', 'Type', 'Elaborate', 'Grapes', 'Harmonize', 'ABV', 'Body', 'Acidity', 'Code', 'Country', 'RegionID', 'RegionName', 'WineryID', 'WineryName', 'Website', 'Vintages']


In [35]:
print(harm_matrix.columns.tolist())

['WineID', 'Pair_Aperitif', 'Pair_Barbecue', 'Pair_Beef', 'Pair_Blue Cheese', 'Pair_Cake', 'Pair_Cheese', 'Pair_Chicken', 'Pair_Chocolate', 'Pair_Codfish', 'Pair_Cold Cuts', 'Pair_Cream', 'Pair_Cured Meat', 'Pair_Dessert', 'Pair_Duck', 'Pair_Fish', 'Pair_French Fries', 'Pair_Fruit', 'Pair_Fruit Dessert', 'Pair_Game Meat', 'Pair_Goat Cheese', 'Pair_Grilled', 'Pair_Ham', 'Pair_Hard Cheese', 'Pair_Lamb', 'Pair_Lean Fish', 'Pair_Light Stews', 'Pair_Maturated Cheese', 'Pair_Mushrooms', 'Pair_Pasta', 'Pair_Pizza', 'Pair_Pork', 'Pair_Poultry', 'Pair_Rich Fish', 'Pair_Risotto', 'Pair_Salad', 'Pair_Seafood', 'Pair_Shellfish', 'Pair_Snack', 'Pair_Soft Cheese', 'Pair_Soufflé', 'Pair_Spicy Food', 'Pair_Sweet Dessert', 'Pair_Tomato Dishes', 'Pair_Veal', 'Pair_Vegetarian']


### Foodpairings info datqsets building

In [36]:
PairType = pd.concat([harm_matrix, type_dummies], axis=1)

# dataset_type_pair = dataset_type_harm.set_index("WineID")

In [37]:
PairType.head()

,WineID,Pair_Aperitif,Pair_Barbecue,Pair_Beef,Pair_Blue Cheese,Pair_Cake,Pair_Cheese,Pair_Chicken,Pair_Chocolate,Pair_Codfish,...,Pair_Sweet Dessert,Pair_Tomato Dishes,Pair_Veal,Pair_Vegetarian,Type_Dessert,Type_Dessert/Port,Type_Red,Type_Rosé,Type_Sparkling,Type_White
0,100001,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,100002,False,True,True,False,False,True,False,False,True,...,False,False,False,False,False,False,True,False,False,False
2,100003,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,100005,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,100007,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False


In [ ]:
PairAcidBody = pd.concat([harm_matrix, acid_dummies, body_dummies], axis=1)

## mlxtend

In [44]:
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

#### Food and  WineType (apriori algo)

In [41]:
freq_PairType = apriori(PairType.drop(columns="WineID"), min_support=0.02, use_colnames=True, low_memory=True)
freq_PairType

,support,itemsets
0,0.101291,(Pair_Aperitif)
1,0.551142,(Pair_Beef)
2,0.022840,(Pair_Blue Cheese)
3,0.114201,(Pair_Cured Meat)
4,0.351539,(Pair_Game Meat)
...,...,...
432,0.025819,"(Type_Red, Pair_Lamb, Pair_Beef, Pair_Game Mea..."
433,0.026812,"(Type_Red, Pair_Lamb, Pair_Beef, Pair_Maturate..."
434,0.025819,"(Pair_Pork, Pair_Spicy Food, Type_White, Pair_..."
435,0.025819,"(Type_Red, Pair_Lamb, Pair_Game Meat, Pair_Mat..."


In [42]:
freq_PairType['length'] = freq_PairType['itemsets'].apply(lambda x: len(x))

In [45]:
freq_PairType.to_csv(f'outputs/mlxtend_X_freq_PairType.csv', index=False)

In [46]:
rules_PairType = association_rules(freq_PairType, metric="confidence", min_threshold=0.6)

In [47]:
rules_PairType

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Pair_Aperitif),(Pair_Shellfish),0.101291,0.254220,0.061569,0.607843,2.391008,1.0,0.035819,1.901738,0.647336,0.209459,0.474165,0.425015
1,(Pair_Snack),(Pair_Aperitif),0.085402,0.101291,0.083416,0.976744,9.642955,1.0,0.074766,38.644489,0.979991,0.807692,0.974123,0.900137
2,(Pair_Aperitif),(Pair_Snack),0.101291,0.085402,0.083416,0.823529,9.642955,1.0,0.074766,5.182721,0.997316,0.807692,0.807051,0.900137
3,(Pair_Game Meat),(Pair_Beef),0.351539,0.551142,0.313803,0.892655,1.619647,1.0,0.120055,4.181467,0.589984,0.532884,0.760849,0.731012
4,(Pair_Hard Cheese),(Pair_Beef),0.116187,0.551142,0.089374,0.769231,1.395703,1.0,0.025339,1.945051,0.320787,0.154639,0.485875,0.465696
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1431,"(Pair_Game Meat, Pair_Maturated Cheese, Pair_H...","(Pair_Beef, Type_Red, Pair_Lamb, Pair_Poultry)",0.034757,0.261172,0.025819,0.742857,2.844324,1.0,0.016742,2.873221,0.671771,0.095588,0.651959,0.420858
1432,"(Pair_Game Meat, Pair_Maturated Cheese, Pair_P...","(Pair_Beef, Type_Red, Pair_Lamb, Pair_Hard Che...",0.029791,0.037736,0.025819,0.866667,22.966667,1.0,0.024695,7.216981,0.985828,0.619048,0.861438,0.775439
1433,"(Pair_Game Meat, Pair_Poultry, Pair_Hard Cheese)","(Pair_Maturated Cheese, Pair_Beef, Type_Red, P...",0.029791,0.042701,0.025819,0.866667,20.296124,1.0,0.024547,7.179742,0.979923,0.553191,0.860719,0.735659
1434,"(Pair_Game Meat, Pair_Maturated Cheese)","(Type_Red, Pair_Lamb, Pair_Beef, Pair_Poultry,...",0.041708,0.026812,0.025819,0.619048,23.088183,1.0,0.024701,2.554618,0.998326,0.604651,0.608552,0.791005


In [48]:
rules_PairType_df1 = rules_PairType[
    rules_PairType['antecedents'].apply(lambda x: any(i in harm_matrix.columns for i in x)) &
    rules_PairType['consequents'].apply(lambda x: any(i in type_dummies.columns for i in x))
]

In [49]:
rules_PairType_df1 

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
12,(Pair_Beef),(Type_Red),0.551142,0.502483,0.467726,0.848649,1.688911,1.0,0.190787,3.287168,0.908757,0.798305,0.695787,0.889739
15,(Pair_Cured Meat),(Type_White),0.114201,0.230387,0.076465,0.669565,2.906259,1.0,0.050154,2.329091,0.740478,0.285185,0.570648,0.500731
19,(Pair_Game Meat),(Type_Red),0.351539,0.502483,0.308838,0.878531,1.748381,1.0,0.132196,4.095841,0.660090,0.566485,0.755850,0.746578
22,(Pair_Goat Cheese),(Type_White),0.042701,0.230387,0.039722,0.930233,4.037690,1.0,0.029884,11.031116,0.785892,0.170213,0.909347,0.551323
27,(Pair_Lamb),(Type_Red),0.406157,0.502483,0.362463,0.892421,1.776023,1.0,0.158376,4.624650,0.735791,0.663636,0.783767,0.806882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1431,"(Pair_Game Meat, Pair_Maturated Cheese, Pair_H...","(Pair_Beef, Type_Red, Pair_Lamb, Pair_Poultry)",0.034757,0.261172,0.025819,0.742857,2.844324,1.0,0.016742,2.873221,0.671771,0.095588,0.651959,0.420858
1432,"(Pair_Game Meat, Pair_Maturated Cheese, Pair_P...","(Pair_Beef, Type_Red, Pair_Lamb, Pair_Hard Che...",0.029791,0.037736,0.025819,0.866667,22.966667,1.0,0.024695,7.216981,0.985828,0.619048,0.861438,0.775439
1433,"(Pair_Game Meat, Pair_Poultry, Pair_Hard Cheese)","(Pair_Maturated Cheese, Pair_Beef, Type_Red, P...",0.029791,0.042701,0.025819,0.866667,20.296124,1.0,0.024547,7.179742,0.979923,0.553191,0.860719,0.735659
1434,"(Pair_Game Meat, Pair_Maturated Cheese)","(Type_Red, Pair_Lamb, Pair_Beef, Pair_Poultry,...",0.041708,0.026812,0.025819,0.619048,23.088183,1.0,0.024701,2.554618,0.998326,0.604651,0.608552,0.791005


In [50]:
rules_PairType_df1.sort_values(by='confidence', ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
479,"(Pair_Pasta, Pair_Cured Meat, Pair_Shellfish)",(Type_White),0.023833,0.230387,0.023833,1.000000,4.340517,1.0,0.018342,inf,0.788403,0.103448,1.000000,0.551724
633,"(Pair_Pasta, Pair_Lamb, Pair_Poultry)",(Type_Red),0.024826,0.502483,0.024826,1.000000,1.990119,1.0,0.012351,inf,0.510183,0.049407,1.000000,0.524704
235,"(Pair_Pasta, Pair_Veal)",(Type_Red),0.036743,0.502483,0.036743,1.000000,1.990119,1.0,0.018280,inf,0.516495,0.073123,1.000000,0.536561
638,"(Pair_Pasta, Pair_Veal, Pair_Poultry)",(Type_Red),0.029791,0.502483,0.029791,1.000000,1.990119,1.0,0.014822,inf,0.512794,0.059289,1.000000,0.529644
141,"(Pair_Pasta, Pair_Cured Meat)",(Type_White),0.029791,0.230387,0.029791,1.000000,4.340517,1.0,0.022928,inf,0.793245,0.129310,1.000000,0.564655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
911,"(Pair_Maturated Cheese, Pair_Poultry)","(Pair_Beef, Type_Red, Pair_Hard Cheese)",0.047666,0.041708,0.028798,0.604167,14.485615,1.0,0.026810,2.420948,0.977563,0.475410,0.586939,0.647321
956,"(Pair_Pork, Pair_Cured Meat, Pair_Poultry)","(Type_White, Pair_Shellfish)",0.047666,0.148957,0.028798,0.604167,4.055972,1.0,0.021698,2.150003,0.791162,0.171598,0.534884,0.398750
667,"(Pair_Pork, Pair_Vegetarian)","(Type_White, Pair_Poultry)",0.092354,0.101291,0.055611,0.602151,5.944761,1.0,0.046256,2.258917,0.916419,0.402878,0.557310,0.575585
617,"(Pair_Maturated Cheese, Pair_Lamb)","(Type_Red, Pair_Hard Cheese)",0.064548,0.043694,0.038729,0.600000,13.731818,1.0,0.035909,2.390765,0.991154,0.557143,0.581724,0.743182


In [51]:
rules_PairType_df1.to_csv(f'outputs/mlxtend_X_rules_PairType.csv', index=False)

In [52]:
rules_PairType_df2 = rules_PairType[
    rules_PairType['antecedents'].apply(lambda x: any(i in type_dummies.columns for i in x)) &
    rules_PairType['consequents'].apply(lambda x: any(i in harm_matrix.columns for i in x))
]

In [54]:
rules_PairType_df2.head(30)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
13,(Type_Red),(Pair_Beef),0.502483,0.551142,0.467726,0.930830,1.688911,1.0,0.190787,6.489204,0.819876,0.798305,0.845898,0.889739
18,(Type_Red),(Pair_Game Meat),0.502483,0.351539,0.308838,0.614625,1.748381,1.0,0.132196,1.682673,0.860356,0.566485,0.405707,0.746578
26,(Type_Red),(Pair_Lamb),0.502483,0.406157,0.362463,0.721344,1.776023,1.0,0.158376,2.131097,0.878249,0.663636,0.530758,0.806882
33,(Type_Sparkling),(Pair_Pork),0.099305,0.262165,0.060576,0.610000,2.326780,1.0,0.034542,1.891885,0.633091,0.201320,0.471427,0.420530
36,(Type_Red),(Pair_Poultry),0.502483,0.577954,0.384310,0.764822,1.323326,1.0,0.093898,1.794581,0.491095,0.552068,0.442767,0.714885
39,(Type_Sparkling),(Pair_Rich Fish),0.099305,0.163853,0.063555,0.640000,3.905939,1.0,0.047284,2.322630,0.826006,0.318408,0.569454,0.513939
42,(Type_Sparkling),(Pair_Shellfish),0.099305,0.254220,0.071500,0.720000,2.832188,1.0,0.046254,2.663498,0.718241,0.253521,0.624554,0.500625
43,(Type_White),(Pair_Shellfish),0.230387,0.254220,0.148957,0.646552,2.543272,1.0,0.090388,2.110010,0.788456,0.443787,0.526069,0.616245
55,"(Type_Sparkling, Pair_Aperitif)",(Pair_Shellfish),0.029791,0.254220,0.023833,0.800000,3.146875,1.0,0.016260,3.728898,0.703173,0.091603,0.731824,0.446875
56,"(Pair_Aperitif, Type_White)",(Pair_Shellfish),0.043694,0.254220,0.027805,0.636364,2.503196,1.0,0.016697,2.050894,0.627948,0.102941,0.512408,0.372869


#### acidity and body apriori algo

In [38]:
PairAcidBody 

,WineID,Pair_Aperitif,Pair_Barbecue,Pair_Beef,Pair_Blue Cheese,Pair_Cake,Pair_Cheese,Pair_Chicken,Pair_Chocolate,Pair_Codfish,...,Pair_Veal,Pair_Vegetarian,Acid_High,Acid_Low,Acid_Medium,Body_Full-bodied,Body_Light-bodied,Body_Medium-bodied,Body_Very full-bodied,Body_Very light-bodied
0,100001,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
1,100002,False,True,True,False,False,True,False,False,True,...,False,False,False,False,True,False,False,True,False,False
2,100003,False,False,True,False,False,False,False,False,False,...,False,False,True,False,False,True,False,False,False,False
3,100005,False,False,True,False,False,False,False,False,False,...,False,False,False,False,True,True,False,False,False,False
4,100007,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1002,199408,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
1003,199481,False,False,True,False,False,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
1004,199533,False,False,False,False,False,False,False,False,False,...,False,True,True,False,False,True,False,False,False,False
1005,199885,False,False,False,False,False,False,False,False,False,...,False,True,True,False,False,False,False,True,False,False


In [55]:
freq_PairAcidBody= apriori(PairAcidBody.drop(columns="WineID"), min_support=0.02, use_colnames=True, low_memory=True)


freq_PairAcidBody

,support,itemsets
0,0.101291,(Pair_Aperitif)
1,0.551142,(Pair_Beef)
2,0.022840,(Pair_Blue Cheese)
3,0.114201,(Pair_Cured Meat)
4,0.351539,(Pair_Game Meat)
...,...,...
918,0.030785,"(Pair_Pork, Acid_High, Pair_Spicy Food, Body_L..."
919,0.026812,"(Pair_Rich Fish, Body_Medium-bodied, Pair_Pork..."
920,0.021847,"(Pair_Lamb, Pair_Beef, Acid_Medium, Pair_Game ..."
921,0.020854,"(Body_Very full-bodied, Pair_Lamb, Pair_Beef, ..."


In [56]:
freq_PairAcidBody.to_csv(f'outputs/mlxtend_X_freq_PairAcidBody.csv', index=False)

In [57]:

rules_PairAcidBody = association_rules(freq_PairAcidBody, metric="confidence", min_threshold=0.6)


In [58]:
rules_PairAcidBody

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Pair_Aperitif),(Pair_Shellfish),0.101291,0.254220,0.061569,0.607843,2.391008,1.0,0.035819,1.901738,0.647336,0.209459,0.474165,0.425015
1,(Pair_Snack),(Pair_Aperitif),0.085402,0.101291,0.083416,0.976744,9.642955,1.0,0.074766,38.644489,0.979991,0.807692,0.974123,0.900137
2,(Pair_Aperitif),(Pair_Snack),0.101291,0.085402,0.083416,0.823529,9.642955,1.0,0.074766,5.182721,0.997316,0.807692,0.807051,0.900137
3,(Pair_Aperitif),(Acid_High),0.101291,0.734856,0.080437,0.794118,1.080644,1.0,0.006003,1.287842,0.083037,0.106439,0.223507,0.451789
4,(Pair_Game Meat),(Pair_Beef),0.351539,0.551142,0.313803,0.892655,1.619647,1.0,0.120055,4.181467,0.589984,0.532884,0.760849,0.731012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3412,"(Pair_Cured Meat, Pair_Shellfish, Pair_Poultry)","(Pair_Pork, Body_Light-bodied, Acid_High, Pair...",0.037736,0.033764,0.030785,0.815789,24.161765,1.0,0.029510,5.245283,0.996205,0.756098,0.809353,0.863777
3413,"(Pair_Pork, Body_Light-bodied)","(Acid_High, Pair_Spicy Food, Pair_Shellfish, P...",0.050645,0.033764,0.030785,0.607843,18.002884,1.0,0.029075,2.463903,0.994837,0.574074,0.594140,0.759804
3414,"(Body_Light-bodied, Pair_Spicy Food)","(Pair_Pork, Acid_High, Pair_Shellfish, Pair_Po...",0.040715,0.037736,0.030785,0.756098,20.036585,1.0,0.029248,3.945283,0.990416,0.645833,0.746533,0.785944
3415,"(Pair_Cured Meat, Pair_Spicy Food)","(Pair_Pork, Acid_High, Body_Light-bodied, Pair...",0.038729,0.034757,0.030785,0.794872,22.869597,1.0,0.029438,4.705561,0.994801,0.720930,0.787485,0.840293


In [59]:
rules_PairAcidBody_df1 = rules_PairAcidBody[
    rules_PairAcidBody ['antecedents'].apply(lambda x: any(i in harm_matrix.columns for i in x)) &
    rules_PairAcidBody['consequents'].apply(lambda x: any(i in acid_dummies.columns or i in body_dummies.columns for i in x))
]

In [60]:
rules_PairAcidBody_df1

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
3,(Pair_Aperitif),(Acid_High),0.101291,0.734856,0.080437,0.794118,1.080644,1.0,0.006003,1.287842,0.083037,0.106439,0.223507,0.451789
13,(Pair_Beef),(Acid_High),0.551142,0.734856,0.394240,0.715315,0.973409,1.0,-0.010770,0.931360,-0.057369,0.442094,-0.073698,0.625901
17,(Pair_Cured Meat),(Acid_High),0.114201,0.734856,0.092354,0.808696,1.100482,1.0,0.008433,1.385980,0.103079,0.122047,0.278489,0.467186
20,(Pair_Game Meat),(Acid_High),0.351539,0.734856,0.240318,0.683616,0.930272,1.0,-0.018013,0.838044,-0.103612,0.284038,-0.193254,0.505321
23,(Pair_Goat Cheese),(Acid_High),0.042701,0.734856,0.041708,0.976744,1.329164,1.0,0.010329,11.401192,0.258694,0.056680,0.912290,0.516750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3412,"(Pair_Cured Meat, Pair_Shellfish, Pair_Poultry)","(Pair_Pork, Body_Light-bodied, Acid_High, Pair...",0.037736,0.033764,0.030785,0.815789,24.161765,1.0,0.029510,5.245283,0.996205,0.756098,0.809353,0.863777
3413,"(Pair_Pork, Body_Light-bodied)","(Acid_High, Pair_Spicy Food, Pair_Shellfish, P...",0.050645,0.033764,0.030785,0.607843,18.002884,1.0,0.029075,2.463903,0.994837,0.574074,0.594140,0.759804
3414,"(Body_Light-bodied, Pair_Spicy Food)","(Pair_Pork, Acid_High, Pair_Shellfish, Pair_Po...",0.040715,0.037736,0.030785,0.756098,20.036585,1.0,0.029248,3.945283,0.990416,0.645833,0.746533,0.785944
3415,"(Pair_Cured Meat, Pair_Spicy Food)","(Pair_Pork, Acid_High, Body_Light-bodied, Pair...",0.038729,0.034757,0.030785,0.794872,22.869597,1.0,0.029438,4.705561,0.994801,0.720930,0.787485,0.840293


In [61]:
rules_PairAcidBody_df1 = rules_PairAcidBody_df1.sort_values(by='confidence', ascending=False)

In [62]:
rules_PairAcidBody_df1.to_csv(f'outputs/mlxtend_X_rules_PairAcidBody.csv', index=False)

### acidity and body  Fp growth

In [63]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Switch to fpgrowth and limit rule length
# FP-Growth is much faster than Apriori. 
# max_len prevents overly complex, overfitted rules (e.g., 6 items -> 1 item).
freq_PairAcidBodyFP = fpgrowth(
    PairAcidBody.drop(columns=["WineID"]), 
    min_support=0.02, 
    use_colnames=True,
    max_len=4 
)

# 2. Generate rules based on Lift, not just Confidence
# A high confidence rule is useless if the lift is 1.0 (meaning it's just a common item).
rules_PairAcidBodyFP = association_rules(
    freq_PairAcidBodyFP, 
    metric="lift", 
    min_threshold=1.5  # Only keep rules with a strong positive correlation
)

# 3. Filter for Wine -> Food recommendations
# We want antecedents to ONLY be wine features, and consequents to ONLY be foods.
wine_cols = set(acid_dummies.columns).union(set(body_dummies.columns))
food_cols = set(harm_matrix.columns)

rules_wine_to_foodFP = rules_PairAcidBodyFP[
    # Antecedents must be a subset of wine columns (Wine features only)
    rules_PairAcidBodyFP['antecedents'].apply(lambda x: set(x).issubset(wine_cols)) &
    
    # Consequents must be a subset of food columns (Food features only)
    rules_PairAcidBodyFP['consequents'].apply(lambda x: set(x).issubset(food_cols)) &
    
    # Optional: ensure high confidence for the final output
    (rules_PairAcidBodyFP['confidence'] >= 0.3)
]

# 4. Sort by Lift to get the most powerful recommendations at the top
rules_wine_to_foodFP = rules_wine_to_foodFP.sort_values(by=['lift', 'confidence'], ascending=[False, False])

In [64]:
rules_wine_to_foodFP.head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2508,"(Body_Very full-bodied, Acid_Medium)","(Pair_Game Meat, Pair_Hard Cheese)",0.038729,0.035750,0.020854,0.538462,15.061966,1.0,0.019469,2.089209,0.971222,0.388889,0.521350,0.560897
1820,"(Body_Very full-bodied, Acid_Medium)","(Pair_Game Meat, Pair_Maturated Cheese)",0.038729,0.041708,0.021847,0.564103,13.525031,1.0,0.020232,2.198434,0.963373,0.372881,0.545131,0.543956
2128,"(Body_Very full-bodied, Acid_Medium)","(Pair_Lamb, Pair_Hard Cheese)",0.038729,0.059583,0.021847,0.564103,9.467521,1.0,0.019539,2.157427,0.930409,0.285714,0.536485,0.465385
1792,"(Body_Very full-bodied, Acid_Medium)","(Pair_Maturated Cheese, Pair_Lamb)",0.038729,0.064548,0.022840,0.589744,9.136489,1.0,0.020340,2.280164,0.926428,0.283951,0.561435,0.471795
3544,"(Acid_High, Body_Light-bodied)","(Pair_Cured Meat, Pair_Spicy Food)",0.090367,0.038729,0.030785,0.340659,8.795999,1.0,0.027285,1.457928,0.974363,0.313131,0.314095,0.567766


In [65]:
rules_wine_to_foodFP.to_csv(f'outputs/mlxtend_X_rules_PairAcidBody_FP.csv', index=False)

#### acidity + body + type

In [67]:
PairTypeAcidBody = pd.concat([harm_matrix, type_dummies, acid_dummies, body_dummies], axis=1)

In [68]:
freq_PairTypeAcidBody = fpgrowth(PairTypeAcidBody.drop(columns="WineID"), 
                                 min_support=0.02, 
                                 use_colnames=True, 
                                 max_len=6)

In [69]:
freq_PairTypeAcidBody

,support,itemsets
0,0.734856,(Acid_High)
1,0.345581,(Body_Medium-bodied)
2,0.262165,(Pair_Pork)
3,0.254220,(Pair_Shellfish)
4,0.163853,(Pair_Rich Fish)
...,...,...
1707,0.038729,"(Pair_Shellfish, Type_White, Pair_Goat Cheese)"
1708,0.038729,"(Pair_Vegetarian, Acid_High, Type_White, Pair_..."
1709,0.038729,"(Pair_Shellfish, Acid_High, Type_White, Pair_G..."
1710,0.038729,"(Pair_Vegetarian, Pair_Shellfish, Type_White, ..."


In [112]:
rules_PairTypeAcidBodyFP = association_rules(
    freq_PairTypeAcidBody, 
    metric="lift", 
    min_threshold=1.5  # Only keep rules with a strong positive correlation
)

# 3. Filter for Wine -> Food recommendations
# We want antecedents to ONLY be wine features, and consequents to ONLY be foods.
wine_cols = set(acid_dummies.columns).union(set(body_dummies.columns)).union(set(type_dummies.columns))
food_cols = set(harm_matrix.columns)

rules_wine_to_foodFP2 = rules_PairTypeAcidBodyFP[
    # Antecedents must be a subset of wine columns (Wine features only)
    rules_PairTypeAcidBodyFP['antecedents'].apply(lambda x: set(x).issubset(wine_cols)) &
    
    # Consequents must be a subset of food columns (Food features only)
    rules_PairTypeAcidBodyFP['consequents'].apply(lambda x: set(x).issubset(food_cols)) &
    
    # Optional: ensure high confidence for the final output
    (rules_PairTypeAcidBodyFP['confidence'] >= 0.3)
]

# 4. Sort by Lift to get the most powerful recommendations at the top
rules_wine_to_foodFP2 = rules_wine_to_foodFP2.sort_values(by=['lift', 'confidence'], ascending=[False, False])

In [113]:
rules_wine_to_foodFP2.to_csv(f'outputs/mlxtend_X_rules_PairTypeAcidBody_FP.csv', index=False)

In [114]:
len(rules_wine_to_foodFP2)

459

In [86]:
len(rules_wine_to_foodFP2)

459

### food to wine

In [115]:
# 1. Define feature sets
wine_cols = set(acid_dummies.columns).union(set(body_dummies.columns)).union(set(type_dummies.columns))
food_cols = set(harm_matrix.columns)

# 2. Filter for Food -> Wine recommendations
rules_food_to_wine = rules_PairAcidBody[
    # Antecedents must be a subset of food columns (Food features only)
    rules_PairAcidBody['antecedents'].apply(lambda x: set(x).issubset(food_cols)) &
    
    # Consequents must be a subset of wine columns (Wine features only)
    rules_PairAcidBody['consequents'].apply(lambda x: set(x).issubset(wine_cols))
]

# 3. Sort by Lift and Confidence to get the most powerful recommendations at the top
rules_food_to_wine = rules_food_to_wine.sort_values(by=['lift', 'confidence'], ascending=[False, False])

# 4. Save to CSV so we can evaluate it
rules_food_to_wine.to_csv("outputs/mlxtend_X_rules_FoodToWine.csv", index=False)
print(f"Total Food -> Wine rules extracted: {len(rules_food_to_wine)}")

Total Food -> Wine rules extracted: 267


## Eval

In [ ]:

df_test = pd.read_csv('data_preprocessed/XWines_Test_100_wines_enriched.csv')



In [99]:
def clean_set(s):
    return str(s).replace("frozenset({", "").replace("})", "").replace("'", "")

# Reusable evaluation function for Top-K precision and recall
def evaluate_top_k(df, k, true_col, pred_col):
    precisions = []
    recalls = []
    for idx, row in df.iterrows():
        true_set = set(row[true_col])
        pred_set = set(row[pred_col][:k])
        
        if len(true_set) > 0 and len(pred_set) > 0:
            intersection = pred_set.intersection(true_set)
            precisions.append(len(intersection) / len(pred_set))
            recalls.append(len(intersection) / len(true_set))
        else:
            precisions.append(0.0)
            recalls.append(0.0)
    avg_precision = sum(precisions) / len(precisions) if precisions else 0
    avg_recall = sum(recalls) / len(recalls) if recalls else 0
    return avg_precision, avg_recall

#### Wine to foods

In [116]:
# Load the finalized FP-Growth rules
df_rules_fp = pd.read_csv('outputs/mlxtend_X_rules_PairTypeAcidBody_FP.csv')

# 1. Parse and filter rules for Wine -> Food
rules_w2f = []
for idx, row in df_rules_fp.iterrows():
    ant = [x.strip() for x in clean_set(row['antecedents']).split(',')]
    con = [x.strip() for x in clean_set(row['consequents']).split(',')]
    
    is_ant_wine = all(x.startswith(('Type_', 'Body_', 'Acid_')) for x in ant) and len(ant) > 0
    is_con_food = all(x.startswith('Pair_') for x in con) and len(con) > 0
    
    if is_ant_wine and is_con_food:
        rules_w2f.append({
            'wine_feats': set(ant),
            'food_feats': set(con),
            'lift': row['lift'],
            'confidence': row['confidence']
        })

print(f"Extracted {len(rules_w2f)} rules for Wine -> Food")

# 2. Generate Predictions
predictions_w2f = []
for idx, wine in df_test.iterrows():
    w_feats = set()
    if pd.notna(wine['Type']): w_feats.add(f"Type_{str(wine['Type']).strip()}")
    if pd.notna(wine['Body']): w_feats.add(f"Body_{str(wine['Body']).strip()}")
    if pd.notna(wine['Acidity']): w_feats.add(f"Acid_{str(wine['Acidity']).strip()}")
    
    try:
        true_foods_raw = ast.literal_eval(wine['Harmonize'])
        true_foods = {f"Pair_{str(x).strip()}" for x in true_foods_raw}
    except:
        true_foods = set()
        
    food_scores = {}
    for r in rules_w2f:
        # Rule antecedent must be a subset of the wine's actual features
        if r['wine_feats'].issubset(w_feats):
            score = r['lift'] * r['confidence']
            for f in r['food_feats']:
                food_scores[f] = food_scores.get(f, 0) + score
                
    predicted_foods = [f for f, s in sorted(food_scores.items(), key=lambda x: x[1], reverse=True)]
    
    predictions_w2f.append({
        'WineName': wine['WineName'],
        'WineFeatures': list(w_feats),
        'True_Foods': list(true_foods),
        'Pred_Foods': predicted_foods,
    })

df_preds_w2f = pd.DataFrame(predictions_w2f)

# 3. Evaluate and display results
p3, r3 = evaluate_top_k(df_preds_w2f, 3, 'True_Foods', 'Pred_Foods')
p5, r5 = evaluate_top_k(df_preds_w2f, 5, 'True_Foods', 'Pred_Foods')

print("\n--- Wine-to-Food Evaluation ---")
print(f"Top 3 - Precision: {p3:.4f}, Recall: {r3:.4f}")
print(f"Top 5 - Precision: {p5:.4f}, Recall: {r5:.4f}")

df_preds_w2f['Pred_Foods_Top5'] = df_preds_w2f['Pred_Foods'].apply(lambda x: x[:5])
display(df_preds_w2f[['WineName', 'WineFeatures', 'True_Foods', 'Pred_Foods_Top5']].head())
# df_preds_w2f.to_csv('predictions_Wine_to_Food.csv', index=False)

Extracted 459 rules for Wine -> Food

--- Wine-to-Food Evaluation ---
Top 3 - Precision: 0.5267, Recall: 0.3730
Top 5 - Precision: 0.4420, Recall: 0.5197


,WineName,WineFeatures,True_Foods,Pred_Foods_Top5
0,Origem Merlot,"[Acid_Medium, Type_Red, Body_Full-bodied]","[Pair_Lamb, Pair_Grilled, Pair_Beef, Pair_Veal...","[Pair_Lamb, Pair_Beef, Pair_Game Meat, Pair_Po..."
1,Reserva Chardonnay,"[Acid_Medium, Body_Medium-bodied, Type_White]","[Pair_Rich Fish, Pair_Vegetarian, Pair_Risotto...","[Pair_Vegetarian, Pair_Shellfish, Pair_Pork, P..."
2,Dona Antonia Porto Reserva Tawny,"[Body_Very full-bodied, Type_Dessert/Port, Aci...","[Pair_Blue Cheese, Pair_Appetizer, Pair_Sweet ...","[Pair_Lamb, Pair_Beef, Pair_Game Meat, Pair_Po..."
3,Fine Ruby Port,"[Body_Very full-bodied, Type_Dessert/Port, Aci...","[Pair_Cake, Pair_Soft Cheese, Pair_Fruit, Pair...","[Pair_Game Meat, Pair_Hard Cheese, Pair_Lamb, ..."
4,Maré Alta,"[Type_White, Body_Very light-bodied, Acid_High]","[Pair_Snack, Pair_Vegetarian, Pair_Appetizer, ...","[Pair_Vegetarian, Pair_Shellfish, Pair_Pork, P..."


In [117]:
df_rules_fp = pd.read_csv('outputs/mlxtend_X_rules_PairTypeAcidBody_FP.csv')

# 1. Parse and filter rules for Wine -> Food
rules_w2f = []
for idx, row in df_rules_fp.iterrows():
    ant = [x.strip() for x in clean_set(row['antecedents']).split(',')]
    con = [x.strip() for x in clean_set(row['consequents']).split(',')]
    
    is_ant_wine = all(x.startswith(('Type_', 'Body_', 'Acid_')) for x in ant) and len(ant) > 0
    is_con_food = all(x.startswith('Pair_') for x in con) and len(con) > 0
    
    if is_ant_wine and is_con_food:
        rules_w2f.append({
            'wine_feats': set(ant),
            'food_feats': set(con),
            'lift': row['lift'],
            'confidence': row['confidence']
        })

print(f"Extracted {len(rules_w2f)} rules for Wine -> Food")

# 2. Generate Predictions and Calculate Row-Level Metrics
predictions_w2f = []
for idx, wine in df_test.iterrows():
    w_feats = set()
    if pd.notna(wine['Type']): w_feats.add(f"Type_{str(wine['Type']).strip()}")
    if pd.notna(wine['Body']): w_feats.add(f"Body_{str(wine['Body']).strip()}")
    if pd.notna(wine['Acidity']): w_feats.add(f"Acid_{str(wine['Acidity']).strip()}")
    
    try:
        true_foods_raw = ast.literal_eval(wine['Harmonize'])
        true_foods = {f"Pair_{str(x).strip()}" for x in true_foods_raw}
    except:
        true_foods = set()
        
    food_scores = {}
    for r in rules_w2f:
        if r['wine_feats'].issubset(w_feats):
            score = r['lift'] * r['confidence']
            for f in r['food_feats']:
                food_scores[f] = food_scores.get(f, 0) + score
                
    # Sort predictions
    predicted_foods = [f for f, s in sorted(food_scores.items(), key=lambda x: x[1], reverse=True)]
    top_5_preds = predicted_foods[:5]
    
    # --- ROW LEVEL EVALUATION (Top 5) ---
    pred_set = set(top_5_preds)
    if len(true_foods) > 0 and len(pred_set) > 0:
        intersection = pred_set.intersection(true_foods)
        row_precision = len(intersection) / len(pred_set)
        row_recall = len(intersection) / len(true_foods)
    else:
        row_precision, row_recall = 0.0, 0.0
    
    # Save to row dictionary
    predictions_w2f.append({
        'WineName': wine['WineName'],
        'WineFeatures': list(w_feats),
        'True_Foods': list(true_foods),
        'Pred_Foods_Top5': top_5_preds,
        'Precision_Top5': round(row_precision, 4),
        'Recall_Top5': round(row_recall, 4)
    })

# 3. Export to CSV
df_preds_w2f = pd.DataFrame(predictions_w2f)
print(f"Overall Average Precision (Top 5): {df_preds_w2f['Precision_Top5'].mean():.4f}")
print(f"Overall Average Recall (Top 5): {df_preds_w2f['Recall_Top5'].mean():.4f}")



Extracted 459 rules for Wine -> Food
Overall Average Precision (Top 5): 0.4420
Overall Average Recall (Top 5): 0.5197


In [118]:
# df_preds_w2f.to_csv('predictions_Wine_to_Food_evaluated.csv', index=False)
display(df_preds_w2f.head())

,WineName,WineFeatures,True_Foods,Pred_Foods_Top5,Precision_Top5,Recall_Top5
0,Origem Merlot,"[Acid_Medium, Type_Red, Body_Full-bodied]","[Pair_Lamb, Pair_Grilled, Pair_Beef, Pair_Veal...","[Pair_Lamb, Pair_Beef, Pair_Game Meat, Pair_Po...",0.6,0.5
1,Reserva Chardonnay,"[Acid_Medium, Body_Medium-bodied, Type_White]","[Pair_Rich Fish, Pair_Vegetarian, Pair_Risotto...","[Pair_Vegetarian, Pair_Shellfish, Pair_Pork, P...",0.2,0.2
2,Dona Antonia Porto Reserva Tawny,"[Body_Very full-bodied, Type_Dessert/Port, Aci...","[Pair_Blue Cheese, Pair_Appetizer, Pair_Sweet ...","[Pair_Lamb, Pair_Beef, Pair_Game Meat, Pair_Po...",0.0,0.0
3,Fine Ruby Port,"[Body_Very full-bodied, Type_Dessert/Port, Aci...","[Pair_Cake, Pair_Soft Cheese, Pair_Fruit, Pair...","[Pair_Game Meat, Pair_Hard Cheese, Pair_Lamb, ...",0.0,0.0
4,Maré Alta,"[Type_White, Body_Very light-bodied, Acid_High]","[Pair_Snack, Pair_Vegetarian, Pair_Appetizer, ...","[Pair_Vegetarian, Pair_Shellfish, Pair_Pork, P...",0.4,0.4


In [119]:
df_preds_w2f.to_csv('outputs/mlxtend_X_wine2food_eval.csv', index=False)
display(df_preds_w2f.head())

,WineName,WineFeatures,True_Foods,Pred_Foods_Top5,Precision_Top5,Recall_Top5
0,Origem Merlot,"[Acid_Medium, Type_Red, Body_Full-bodied]","[Pair_Lamb, Pair_Grilled, Pair_Beef, Pair_Veal...","[Pair_Lamb, Pair_Beef, Pair_Game Meat, Pair_Po...",0.6,0.5
1,Reserva Chardonnay,"[Acid_Medium, Body_Medium-bodied, Type_White]","[Pair_Rich Fish, Pair_Vegetarian, Pair_Risotto...","[Pair_Vegetarian, Pair_Shellfish, Pair_Pork, P...",0.2,0.2
2,Dona Antonia Porto Reserva Tawny,"[Body_Very full-bodied, Type_Dessert/Port, Aci...","[Pair_Blue Cheese, Pair_Appetizer, Pair_Sweet ...","[Pair_Lamb, Pair_Beef, Pair_Game Meat, Pair_Po...",0.0,0.0
3,Fine Ruby Port,"[Body_Very full-bodied, Type_Dessert/Port, Aci...","[Pair_Cake, Pair_Soft Cheese, Pair_Fruit, Pair...","[Pair_Game Meat, Pair_Hard Cheese, Pair_Lamb, ...",0.0,0.0
4,Maré Alta,"[Type_White, Body_Very light-bodied, Acid_High]","[Pair_Snack, Pair_Vegetarian, Pair_Appetizer, ...","[Pair_Vegetarian, Pair_Shellfish, Pair_Pork, P...",0.4,0.4


In [ ]:
df_rules_fp = pd.read_csv('mlxtend_X_rules_PairTypeAcidBody_FP.csv')

# 1. Parse and filter rules for Wine -> Food
rules_w2f = []
for idx, row in df_rules_fp.iterrows():
    ant = [x.strip() for x in clean_set(row['antecedents']).split(',')]
    con = [x.strip() for x in clean_set(row['consequents']).split(',')]
    
    is_ant_wine = all(x.startswith(('Type_', 'Body_', 'Acid_')) for x in ant) and len(ant) > 0
    is_con_food = all(x.startswith('Pair_') for x in con) and len(con) > 0
    
    if is_ant_wine and is_con_food:
        rules_w2f.append({
            'wine_feats': set(ant),
            'food_feats': set(con),
            'lift': row['lift'],
            'confidence': row['confidence']
        })

print(f"Extracted {len(rules_w2f)} rules for Wine -> Food")

# 2. Generate Predictions and Calculate Row-Level Metrics
predictions_w2f = []
for idx, wine in df_wines.iterrows():
    w_feats = set()
    if pd.notna(wine['Type']): w_feats.add(f"Type_{str(wine['Type']).strip()}")
    if pd.notna(wine['Body']): w_feats.add(f"Body_{str(wine['Body']).strip()}")
    if pd.notna(wine['Acidity']): w_feats.add(f"Acid_{str(wine['Acidity']).strip()}")
    
    try:
        true_foods_raw = ast.literal_eval(wine['Harmonize'])
        true_foods = {f"Pair_{str(x).strip()}" for x in true_foods_raw}
    except:
        true_foods = set()
        
    food_scores = {}
    for r in rules_w2f:
        if r['wine_feats'].issubset(w_feats):
            score = r['lift'] * r['confidence']
            for f in r['food_feats']:
                food_scores[f] = food_scores.get(f, 0) + score
                
    # Sort predictions
    predicted_foods = [f for f, s in sorted(food_scores.items(), key=lambda x: x[1], reverse=True)]
    top_5_preds = predicted_foods[:5]
    
    # --- ROW LEVEL EVALUATION (Top 5) ---
    pred_set = set(top_5_preds)
    if len(true_foods) > 0 and len(pred_set) > 0:
        intersection = pred_set.intersection(true_foods)
        row_precision = len(intersection) / len(pred_set)
        row_recall = len(intersection) / len(true_foods)
    else:
        row_precision, row_recall = 0.0, 0.0
    
    # Save to row dictionary
    predictions_w2f.append({
        'WineName': wine['WineName'],
        'WineFeatures': list(w_feats),
        'True_Foods': list(true_foods),
        'Pred_Foods_Top5': top_5_preds,
        'Precision_Top5': round(row_precision, 4),
        'Recall_Top5': round(row_recall, 4)
    })

# 3. Export to CSV
df_preds_w2f = pd.DataFrame(predictions_w2f)
print(f"Overall Average Precision (Top 5): {df_preds_w2f['Precision_Top5'].mean():.4f}")
print(f"Overall Average Recall (Top 5): {df_preds_w2f['Recall_Top5'].mean():.4f}")

df_preds_w2f.to_csv('predictions_Wine_to_Food_evaluated.csv', index=False)
display(df_preds_w2f.head())

#### Food to wine

In [101]:
df_rules = pd.read_csv('outputs/mlxtend_X_rules_FoodToWine.csv')



# 2. Parse the rules into a list of dictionaries
parsed_rules = []
for idx, row in df_rules.iterrows():
    ant = {x.strip() for x in clean_set(row['antecedents']).split(',')}
    con = {x.strip() for x in clean_set(row['consequents']).split(',')}
    parsed_rules.append({
        'food_feats': ant,
        'wine_feats': con,
        'lift': row['lift'],
        'confidence': row['confidence']
    })

In [103]:
predictions = []

# 3. Iterate through the test set
for idx, wine in df_test.iterrows():
    # The true wine profile we are trying to predict (Up to 3 features)
    true_wine_profile = set()
    if pd.notna(wine['Type']): true_wine_profile.add(f"Type_{str(wine['Type']).strip()}")
    if pd.notna(wine['Body']): true_wine_profile.add(f"Body_{str(wine['Body']).strip()}")
    if pd.notna(wine['Acidity']): true_wine_profile.add(f"Acid_{str(wine['Acidity']).strip()}")
    
    # The foods from the Harmonize column (This is our user INPUT)
    try:
        input_foods = {f"Pair_{str(x).strip()}" for x in ast.literal_eval(wine['Harmonize'])}
    except:
        input_foods = set()
        
    wine_feature_scores = {}
    
    # 4. Match the input meal to the rules
    for r in parsed_rules:
        # If the rule requires a food combination that is fully present in our input meal
        if r['food_feats'].issubset(input_foods) and len(r['food_feats']) > 0:
            score = r['lift'] * r['confidence']
            for w_feat in r['wine_feats']:
                wine_feature_scores[w_feat] = wine_feature_scores.get(w_feat, 0) + score
                
    # 5. Rank the predicted wine features by their score
    ranked_predictions = [feat for feat, score in sorted(wine_feature_scores.items(), key=lambda x: x[1], reverse=True)]
    
    predictions.append({
        'WineName': wine['WineName'],
        'Input_Meal': list(input_foods),
        'True_Wine_Profile': list(true_wine_profile),
        'Predicted_Top3_Profile': ranked_predictions[:3] # We only care about the top 3 (Type, Body, Acid)
    })

df_preds = pd.DataFrame(predictions)

# 6. Evaluate how many of the top 3 predictions actually matched the real wine profile
precisions = []
for idx, row in df_preds.iterrows():
    true_set = set(row['True_Wine_Profile'])
    pred_set = set(row['Predicted_Top3_Profile'])
    
    if len(true_set) > 0 and len(pred_set) > 0:
        intersection = pred_set.intersection(true_set)
        # Precision = Correctly predicted features / Total predicted features
        precisions.append(len(intersection) / len(pred_set))
    else:
        precisions.append(0.0)

# Print final evaluation metrics
print(f"Overall Precision for Wine Profile Prediction: {(sum(precisions)/len(precisions)) * 100:.4f}%")

# Print some samples to inspect
print("\nSample Predictions (Meal -> Wine):")
print(df_preds[['Input_Meal', 'True_Wine_Profile', 'Predicted_Top3_Profile']].head(10).to_string())
df_preds.to_csv('outputs/mlxtend_X_food2wine_eval.csv', index=False)

Overall Precision for Wine Profile Prediction: 52.0000%

Sample Predictions (Meal -> Wine):
                                                                    Input_Meal                                        True_Wine_Profile                           Predicted_Top3_Profile
0      [Pair_Lamb, Pair_Grilled, Pair_Beef, Pair_Veal, Pair_Pasta, Pair_Pizza]                [Acid_Medium, Type_Red, Body_Full-bodied]                    [Acid_High, Body_Full-bodied]
1  [Pair_Rich Fish, Pair_Vegetarian, Pair_Risotto, Pair_Seafood, Pair_Poultry]            [Acid_Medium, Body_Medium-bodied, Type_White]                                      [Acid_High]
2                       [Pair_Blue Cheese, Pair_Appetizer, Pair_Sweet Dessert]    [Body_Very full-bodied, Type_Dessert/Port, Acid_High]                                      [Acid_High]
3                [Pair_Cake, Pair_Soft Cheese, Pair_Fruit, Pair_Sweet Dessert]  [Body_Very full-bodied, Type_Dessert/Port, Acid_Medium]                                 

In [107]:

# 1. Load your new Food->Wine rules and the 100 wines test set
df_rules = pd.read_csv('outputs/mlxtend_X_rules_FoodToWine.csv')

def clean_set(s):
    return str(s).replace("frozenset({", "").replace("})", "").replace("'", "")

# 2. Parse the rules into a list of dictionaries
parsed_rules = []
for idx, row in df_rules.iterrows():
    ant = {x.strip() for x in clean_set(row['antecedents']).split(',')}
    con = {x.strip() for x in clean_set(row['consequents']).split(',')}
    parsed_rules.append({
        'food_feats': ant,
        'wine_feats': con,
        'lift': row['lift'],
        'confidence': row['confidence']
    })

predictions = []

# 3. Iterate through the test set
for idx, wine in df_test.iterrows():
    # The true wine profile we are trying to predict (Up to 3 features)
    true_wine_profile = set()
    if pd.notna(wine['Type']): true_wine_profile.add(f"Type_{str(wine['Type']).strip()}")
    if pd.notna(wine['Body']): true_wine_profile.add(f"Body_{str(wine['Body']).strip()}")
    if pd.notna(wine['Acidity']): true_wine_profile.add(f"Acid_{str(wine['Acidity']).strip()}")
    
    # The foods from the Harmonize column (This is our user INPUT)
    try:
        input_foods = {f"Pair_{str(x).strip()}" for x in ast.literal_eval(wine['Harmonize'])}
    except:
        input_foods = set()
        
    wine_feature_scores = {}
    
    # 4. Match the input meal to the rules
    for r in parsed_rules:
        # If the rule requires a food combination that is fully present in our input meal
        if r['food_feats'].issubset(input_foods) and len(r['food_feats']) > 0:
            score = r['lift'] * r['confidence']
            for w_feat in r['wine_feats']:
                wine_feature_scores[w_feat] = wine_feature_scores.get(w_feat, 0) + score
                
    # 5. Rank the predicted wine features by their score
    ranked_predictions = [feat for feat, score in sorted(wine_feature_scores.items(), key=lambda x: x[1], reverse=True)]
    
    predictions.append({
        'WineName': wine['WineName'],
        'Input_Meal': list(input_foods),
        'True_Wine_Profile': list(true_wine_profile),
        'Predicted_Top3_Profile': ranked_predictions[:3] # We only care about the top 3 (Type, Body, Acid)
    })

df_preds = pd.DataFrame(predictions)

# 6. Evaluate how many of the top 3 predictions actually matched the real wine profile
precisions = []
for idx, row in df_preds.iterrows():
    true_set = set(row['True_Wine_Profile'])
    pred_set = set(row['Predicted_Top3_Profile'])
    
    if len(true_set) > 0 and len(pred_set) > 0:
        intersection = pred_set.intersection(true_set)
        # Precision = Correctly predicted features / Total predicted features
        precisions.append(len(intersection) / len(pred_set))
    else:
        precisions.append(0.0)

# Print final evaluation metrics
print(f"Overall Precision for Wine Profile Prediction: {(sum(precisions)/len(precisions)) * 100:.2f}%")

# Print some samples to inspect
print("\nSample Predictions (Meal -> Wine):")
print(df_preds[['Input_Meal', 'True_Wine_Profile', 'Predicted_Top3_Profile']].head(10).to_string())
# df_preds.to_csv('food_to_wine_predictions.csv', index=False)

Overall Precision for Wine Profile Prediction: 52.00%

Sample Predictions (Meal -> Wine):
                                                                    Input_Meal                                        True_Wine_Profile                           Predicted_Top3_Profile
0      [Pair_Lamb, Pair_Grilled, Pair_Beef, Pair_Veal, Pair_Pasta, Pair_Pizza]                [Acid_Medium, Type_Red, Body_Full-bodied]                    [Acid_High, Body_Full-bodied]
1  [Pair_Rich Fish, Pair_Vegetarian, Pair_Risotto, Pair_Seafood, Pair_Poultry]            [Acid_Medium, Body_Medium-bodied, Type_White]                                      [Acid_High]
2                       [Pair_Blue Cheese, Pair_Appetizer, Pair_Sweet Dessert]    [Body_Very full-bodied, Type_Dessert/Port, Acid_High]                                      [Acid_High]
3                [Pair_Cake, Pair_Soft Cheese, Pair_Fruit, Pair_Sweet Dessert]  [Body_Very full-bodied, Type_Dessert/Port, Acid_Medium]                                   

In [120]:

def clean_set(s):
    """Helper function to strip frozenset formatting."""
    return str(s).replace("frozenset({", "").replace("})", "").replace("'", "")

def run_experiment(rules_file, test_wines_file, direction='w2f', top_k_w2f=5, top_k_f2w=3):
    """
    Runs a prediction experiment using a specified rules dataset.
    
    Parameters:
    - rules_file: path to the mlxtend rules CSV
    - test_wines_file: path to the XWines 100 test dataset
    - direction: 'w2f' (Wine to Food) or 'f2w' (Food to Wine)
    - top_k_w2f: How many top foods to predict (default: 5)
    - top_k_f2w: How many top wine features to predict (default: 3)
    
    Returns:
    - A Pandas DataFrame containing the predictions and row-level Precision/Recall.
    """
    df_rules = pd.read_csv(rules_file)
    df_wines = pd.read_csv(test_wines_file)
    
    # 1. Dynamically identify which Wine features are present in this specific dataset
    all_rule_items = set()
    for idx, row in df_rules.head(100).iterrows(): # Sample first 100 rules
        all_rule_items.update([x.strip() for x in clean_set(row['antecedents']).split(',')])
        all_rule_items.update([x.strip() for x in clean_set(row['consequents']).split(',')])
        
    has_type = any(x.startswith('Type_') for x in all_rule_items)
    has_body = any(x.startswith('Body_') for x in all_rule_items)
    has_acid = any(x.startswith('Acid_') for x in all_rule_items)
    
    wine_prefixes = []
    if has_type: wine_prefixes.append('Type_')
    if has_body: wine_prefixes.append('Body_')
    if has_acid: wine_prefixes.append('Acid_')
    wine_prefixes = tuple(wine_prefixes)

    # 2. Parse and filter rules based on the desired direction
    rules = []
    for idx, row in df_rules.iterrows():
        ant = [x.strip() for x in clean_set(row['antecedents']).split(',')]
        con = [x.strip() for x in clean_set(row['consequents']).split(',')]
        
        is_ant_wine = all(x.startswith(wine_prefixes) for x in ant) and len(ant) > 0
        is_ant_food = all(x.startswith('Pair_') for x in ant) and len(ant) > 0
        is_con_wine = all(x.startswith(wine_prefixes) for x in con) and len(con) > 0
        is_con_food = all(x.startswith('Pair_') for x in con) and len(con) > 0
        
        if direction == 'w2f' and is_ant_wine and is_con_food:
            rules.append({'wine_feats': set(ant), 'food_feats': set(con), 'lift': row['lift'], 'confidence': row['confidence']})
        elif direction == 'f2w' and is_ant_food and is_con_wine:
            rules.append({'food_feats': set(ant), 'wine_feats': set(con), 'lift': row['lift'], 'confidence': row['confidence']})

    print(f"[{rules_file} | {direction.upper()}] Extracted {len(rules)} valid rules.")

    # 3. Generate Predictions & Calculate Row-Level Metrics
    predictions = []
    for idx, wine in df_wines.iterrows():
        # Build the actual True Wine Profile dynamically
        w_feats = set()
        if has_type and pd.notna(wine['Type']): w_feats.add(f"Type_{str(wine['Type']).strip()}")
        if has_body and pd.notna(wine['Body']): w_feats.add(f"Body_{str(wine['Body']).strip()}")
        if has_acid and pd.notna(wine['Acidity']): w_feats.add(f"Acid_{str(wine['Acidity']).strip()}")
        
        try:
            true_foods_raw = ast.literal_eval(wine['Harmonize'])
            true_foods = {f"Pair_{str(x).strip()}" for x in true_foods_raw}
        except:
            true_foods = set()
            
        scores = {}
        
        if direction == 'w2f':
            for r in rules:
                if r['wine_feats'].issubset(w_feats) and len(r['wine_feats']) > 0:
                    score = r['lift'] * r['confidence']
                    for f in r['food_feats']:
                        scores[f] = scores.get(f, 0) + score
            
            top_preds = [f for f, s in sorted(scores.items(), key=lambda x: x[1], reverse=True)][:top_k_w2f]
            
            pred_set = set(top_preds)
            if len(true_foods) > 0 and len(pred_set) > 0:
                intersection = pred_set.intersection(true_foods)
                prec, rec = len(intersection)/len(pred_set), len(intersection)/len(true_foods)
            else:
                prec, rec = 0.0, 0.0
                
            predictions.append({
                'WineName': wine['WineName'],
                'True_Foods': list(true_foods),
                'Pred_Foods': top_preds,
                f'Precision_Top{top_k_w2f}': round(prec, 4),
                f'Recall_Top{top_k_w2f}': round(rec, 4)
            })
            
        elif direction == 'f2w':
            for r in rules:
                if r['food_feats'].issubset(true_foods) and len(r['food_feats']) > 0:
                    score = r['lift'] * r['confidence']
                    for w in r['wine_feats']:
                        scores[w] = scores.get(w, 0) + score
            
            top_preds = [w for w, s in sorted(scores.items(), key=lambda x: x[1], reverse=True)][:top_k_f2w]
            
            pred_set = set(top_preds)
            if len(w_feats) > 0 and len(pred_set) > 0:
                intersection = pred_set.intersection(w_feats)
                prec, rec = len(intersection)/len(pred_set), len(intersection)/len(w_feats)
            else:
                prec, rec = 0.0, 0.0
                
            predictions.append({
                'WineName': wine['WineName'],
                'Input_Foods': list(true_foods),
                'True_Wine': list(w_feats),
                'Pred_Wine': top_preds,
                f'Precision_Top{top_k_f2w}': round(prec, 4),
                f'Recall_Top{top_k_f2w}': round(rec, 4)
            })
            
    df_preds = pd.DataFrame(predictions)
    
    # Calculate Overall Performance
    if len(df_preds) > 0:
        prec_col = f'Precision_Top{top_k_w2f}' if direction == 'w2f' else f'Precision_Top{top_k_f2w}'
        rec_col = f'Recall_Top{top_k_w2f}' if direction == 'w2f' else f'Recall_Top{top_k_f2w}'
        print(f"--> Avg Precision: {df_preds[prec_col].mean():.4f} | Avg Recall: {df_preds[rec_col].mean():.4f}\n")
    
    return df_preds

In [125]:
test_file = 'data_preprocessed/XWines_Test_100_wines_enriched.csv'

# Add any dataset you want to test to this list
rule_datasets = [
    'mlxtend_X_rules_PairAcidBody.csv',
    'mlxtend_X_rules_PairAcidBody_FP.csv',
    'mlxtend_X_rules_PairType.csv',
    'mlxtend_X_rules_PairTypeAcidBody_FP.csv'
]

results = {}

for rules_csv in rule_datasets:
    print(f"Testing dataset: {rules_csv}")
    rules_csv = "outputs/" + rules_csv if not rules_csv.startswith("outputs/") else rules_csv
    
    # 1. Test Wine-to-Food (Predict 5 Foods)
    df_w2f = run_experiment(rules_csv, test_file, direction='w2f', top_k_w2f=5)
    if len(df_w2f) > 0:
        df_w2f.to_csv(f"{rules_csv.replace('.csv', '')}_W2F_eval.csv", index=False)
        results[f"{rules_csv}_w2f"] = df_w2f
        
    # 2. Test Food-to-Wine (Predict 3 Wine Features)
    df_f2w = run_experiment(rules_csv, test_file, direction='f2w', top_k_f2w=3)
    if len(df_f2w) > 0:
        df_f2w.to_csv(f"{rules_csv.replace('.csv', '')}_F2W_eval.csv", index=False)
        results[f"{rules_csv}_f2w"] = df_f2w

Testing dataset: mlxtend_X_rules_PairAcidBody.csv
[outputs/mlxtend_X_rules_PairAcidBody.csv | W2F] Extracted 0 valid rules.
--> Avg Precision: 0.0000 | Avg Recall: 0.0000

[outputs/mlxtend_X_rules_PairAcidBody.csv | F2W] Extracted 267 valid rules.
--> Avg Precision: 0.5200 | Avg Recall: 0.4700

Testing dataset: mlxtend_X_rules_PairAcidBody_FP.csv
[outputs/mlxtend_X_rules_PairAcidBody_FP.csv | W2F] Extracted 74 valid rules.
--> Avg Precision: 0.4150 | Avg Recall: 0.3318

[outputs/mlxtend_X_rules_PairAcidBody_FP.csv | F2W] Extracted 0 valid rules.
--> Avg Precision: 0.0000 | Avg Recall: 0.0000

Testing dataset: mlxtend_X_rules_PairType.csv
[outputs/mlxtend_X_rules_PairType.csv | W2F] Extracted 0 valid rules.
--> Avg Precision: 0.0000 | Avg Recall: 0.0000

[outputs/mlxtend_X_rules_PairType.csv | F2W] Extracted 134 valid rules.
--> Avg Precision: 0.6450 | Avg Recall: 0.7000

Testing dataset: mlxtend_X_rules_PairTypeAcidBody_FP.csv
[outputs/mlxtend_X_rules_PairTypeAcidBody_FP.csv | W2F] Ext